# 22 — Multi-Distance Bayesian Calibration (`autocalibrate_multi_bayesian`)

Notebook **07** showed analytically that a *single* calibrant distance
cannot break the `(L_sd, a)` degeneracy, and that **two distances** do.
This notebook runs the production multi-image entry point on **two
synthetic CeO2 images at different L_sd**, then computes the **Laplace
(Cramér-Rao) covariance at the MAP** — combining
`autocalibrate_multi` with the Fisher-at-MAP machinery in one call:
`autocalibrate_multi_bayesian`.

Self-contained synthetic data; runtime ~35 s on a CPU.


In [1]:
import os, time
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')   # macOS OpenMP guard
import numpy as np
import torch

from midas_integrate.geometry import build_tilt_matrix, pixel_to_REta
from midas_calibrate import CalibrationParams, build_ring_table


def make_truth(Lsd_um=1_000_000.0) -> CalibrationParams:
    """Known-truth CeO2 geometry on a small 1024x1024 detector."""
    p = CalibrationParams()
    p.NrPixelsY = 1024; p.NrPixelsZ = 1024
    p.pxY = 200.0; p.pxZ = 200.0
    p.Lsd = Lsd_um
    p.BC_y = 512.0; p.BC_z = 512.0
    p.tx = 0.0; p.ty = 0.4; p.tz = 0.25
    p.Wavelength = 0.173
    p.SpaceGroup = 225
    p.LatticeConstant = (5.411, 5.411, 5.411, 90.0, 90.0, 90.0)
    p.MaxRingRad = 480.0
    p.MinRingRad = 0.0
    p.RhoD = 512.0
    p.Width = 1500.0
    p.EtaBinSize = 10.0
    p.RBinSize = 1.0
    p.nIterations = 4
    p.RemoveOutliersBetweenIters = False
    p.SNRMin = 1.5
    p.tolLsd = 5000.0; p.tolBC = 8.0; p.tolTilts = 1.0
    p.tolDistortion = 0.0
    p.Refine = {
        'Lsd': True, 'BC': True, 'ty': True, 'tz': True,
        'Wavelength': False, 'Parallax': False,
        **{f'p{i}': False for i in range(15)},
    }
    return p


def simulate_image(params, ring_thickness_px: float = 1.5) -> np.ndarray:
    """Render a 2D image: bright Gaussian rings on a noisy background."""
    rt = build_ring_table(params)
    NY, NZ = params.NrPixelsY, params.NrPixelsZ
    px = 0.5 * (params.pxY + params.pxZ)
    TRs = build_tilt_matrix(params.tx, params.ty, params.tz)
    Y_grid, Z_grid = np.meshgrid(np.arange(NY, dtype=np.float64),
                                 np.arange(NZ, dtype=np.float64))
    R_pix, _ = pixel_to_REta(
        Y_grid, Z_grid, Ycen=params.BC_y, Zcen=params.BC_z, TRs=TRs,
        Lsd=params.Lsd, RhoD=params.RhoD, px=px, parallax=params.Parallax)
    img = np.full(R_pix.shape, 50.0, dtype=np.float64)
    rng = np.random.default_rng(0)
    img += rng.normal(0, 5.0, size=img.shape)
    for r_ideal in rt.r_ideal_px:
        I_amp = 1000.0 / (1.0 + r_ideal / 100.0)
        img += I_amp * np.exp(-0.5 * ((R_pix - r_ideal) / ring_thickness_px) ** 2)
    return img


## Render two CeO2 images at distinct L_sd

Same beam centre, tilts, and calibrant — only the sample-detector
distance differs (1000 mm vs 1300 mm).


In [2]:
truth1 = make_truth(Lsd_um=1_000_000.0)
truth2 = make_truth(Lsd_um=1_300_000.0)
img1 = simulate_image(truth1)
img2 = simulate_image(truth2)

seed1 = make_truth(Lsd_um=1_000_000.0); seed1.Lsd += 300.0
seed2 = make_truth(Lsd_um=1_300_000.0); seed2.Lsd += 300.0
print(f'image 1: Lsd={truth1.Lsd:.0f} um   image 2: Lsd={truth2.Lsd:.0f} um')


image 1: Lsd=1000000 um   image 2: Lsd=1300000 um


## Build the multi-image spec

`build_multi_spec` creates a spec where the detector + calibrant are
shared and the geometry that genuinely differs between images (here
`Lsd`) is per-image.  The flat refined-name layout is shared-block
first, then per-image with `_imgK` suffixes — the order
`LaplaceResult.sigma_per_dim` inherits.


In [3]:
import torch
from midas_calibrate_v2.pipelines.multi import build_multi_spec
from midas_calibrate_v2.pipelines.bayesian_multi import (
    autocalibrate_multi_bayesian, _flat_refined_names,
)
from midas_calibrate_v2.parameters.pack import pack_multi

multi_spec = build_multi_spec([seed1, seed2])
_, info = pack_multi(multi_spec, dtype=torch.float64, device='cpu')
names = _flat_refined_names(multi_spec, info)
print('refined names:', names)


DIPlib -- a quantitative image analysis library
Version 3.5.2 (Dec 27 2024)
For more information see https://diplib.org
refined names: ['Lsd_img0', 'BC_y_img0', 'BC_z_img0', 'ty_img0', 'tz_img0', 'Lsd_img1', 'BC_y_img1', 'BC_z_img1', 'ty_img1', 'tz_img1']


/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Run MAP + Laplace in one call

`autocalibrate_multi_bayesian` first runs the alternating LM across
both images (the MAP), then evaluates the Fisher at MAP for the
per-parameter σ.


In [4]:
t0 = time.time()
res = autocalibrate_multi_bayesian(
    [seed1, seed2], [img1, img2],
    multi_spec=multi_spec,
    n_iter_map=2, lm_max_iter=80,
    method='fisher',
    verbose=False,
)
print(f'multi-distance MAP + Laplace done in {time.time()-t0:.1f}s')
print(f'joint cost = {res.multi_result.cost:.4e}  rc={res.multi_result.rc}')


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/Context.cpp:767.)
  return torch.sparse_csr_tensor(indptr, indices, values,
/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:51.)
  return torch.sparse_csr_tensor(indptr, in

multi-distance MAP + Laplace done in 25.7s
joint cost = 1.8928e-05  rc=0


## Per-image L_sd recovery vs truth


In [5]:
per = res.multi_result.per_image_unpacked
for k, (truth, p) in enumerate(zip([truth1, truth2], per)):
    Lsd_map = float(p['Lsd'])
    print(f'image {k}: truth Lsd={truth.Lsd:.1f}  MAP={Lsd_map:.1f}  '
          f'err={Lsd_map-truth.Lsd:+.1f} um')


image 0: truth Lsd=1000000.0  MAP=999986.7  err=-13.3 um
image 1: truth Lsd=1300000.0  MAP=1299854.0  err=-146.0 um


## Per-parameter Bayesian σ (Laplace at MAP)

The headline Bayesian output: a calibrated σ on every refined
parameter, aligned to `refined_names`.


In [6]:
lap = res.laplace
print(f'{"parameter":<14s}  {"sigma":>12s}')
for n, s in zip(lap.refined_names, lap.sigma_per_dim):
    unit = 'um' if n.startswith('Lsd') else ('px' if n.startswith('BC') else 'deg')
    print(f'{n:<14s}  {float(s):>12.4e}  {unit}')


parameter              sigma
Lsd_img0          2.0175e+01  um
BC_y_img0         1.9621e-02  px
BC_z_img0         1.9622e-02  px
ty_img0           3.2262e-02  deg
tz_img0           3.2264e-02  deg
Lsd_img1          3.5540e+01  um
BC_y_img1         3.8781e-02  px
BC_z_img1         3.8675e-02  px
ty_img1           7.4187e-02  deg
tz_img1           7.4691e-02  deg


## Why two distances

* A **single** distance leaves `(L_sd, a)` degenerate at small 2θ (see
  notebook 07's analytical Fisher).  Adding a tight L_sd *prior* does
  **not** help — the degeneracy is only broken by an *independent*
  constraint at a *different* L_sd.
* The two-distance protocol adds an independent `Lsd_img1` direction to
  the Fisher, so the shared calibrant lattice constant — and each
  image's L_sd — become identifiable, with the σ values quoted above.

## Practical recipe

Take two CeO₂ images at distinct L_sd at the start of an experimental
session and run them through `autocalibrate_multi_bayesian`.  The σ it
returns is the appropriate uncertainty to quote when reporting absolute
lattice constants or strain downstream.

## See also

- **07_multi_distance** — the analytical `(L_sd, a)` Fisher and why a
  prior alone won't break the gauge.
- **02_bayesian_uncertainty** — single-image Laplace σ.
- `midas_joint_ff_calibrate` notebook 04 — the analogous `(L_sd, λ)`
  gauge broken by adding a powder modality.
